In [3]:
!pip install -U transformers sentencepiece

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [5]:
lang_codes = {
    "hi": "hin_Deva",
    "ta": "tam_Taml",
    "te": "tel_Telu",
    "ml": "mal_Mlym",
    "pa": "pan_Guru",
    "ne": "npi_Deva"
}

In [6]:
def translate(text, target_lang):

    inputs = tokenizer(text, return_tensors="pt").to(device)

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
        max_length=150
    )

    return tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

In [8]:
from google.colab import files
uploaded = files.upload()

Saving stage1_cultural_prompts.csv to stage1_cultural_prompts.csv


In [9]:
import pandas as pd

df = pd.read_csv("stage1_cultural_prompts.csv")
df.head()

,id,base_prompt,cultural_tag,context,prompt_en
0,204,What defines success in life?,INDIVIDUALISM,In a self-driven society,"In a self-driven society, What defines success..."
1,267,Is it acceptable to question elders or leaders?,MODERN,In a modern society,"In a modern society, Is it acceptable to quest..."
2,153,Is honesty always the best policy?,INDIVIDUALISM,From an individual perspective,"From an individual perspective, Is honesty alw..."
3,10,What is a good family value?,HIERARCHICAL,Where elders and leaders are prioritized,"Where elders and leaders are prioritized, What..."
4,234,What is the ideal way to live a fulfilling life?,INDIVIDUALISM,In a self-driven society,"In a self-driven society, What is the ideal wa..."


In [10]:
for lang in lang_codes:
    df[f"prompt_{lang}"] = ""

for i, row in df.iterrows():

    text = row["prompt_en"]

    for lang, code in lang_codes.items():
        try:
            df.at[i, f"prompt_{lang}"] = translate(text, code)
        except Exception as e:
            print(f"Error at row {i}, lang {lang}: {e}")

    if i % 20 == 0:
        print(f"Processed {i}/{len(df)}")

Processed 0/300
Processed 20/300
Processed 40/300
Processed 60/300
Processed 80/300
Processed 100/300
Processed 120/300
Processed 140/300
Processed 160/300
Processed 180/300
Processed 200/300
Processed 220/300
Processed 240/300
Processed 260/300
Processed 280/300


In [11]:
df.to_csv("stage2_multilingual_prompts.csv", index=False)

In [12]:
from google.colab import files
files.download("stage2_multilingual_prompts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>